In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import joblib
import os

In [3]:
# Load data
print("Loading data...")
df = pd.read_csv('../data/processed/clean_data.csv', parse_dates=['InvoiceDate'])

df.head()

Loading data...


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Total_Spend
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34


In [4]:
# Clean & prepare 
print("Preparing data...")

# 1. Clean StockCode: Ensure they are strings and remove whitespace
# This prevents "85123A" and "85123A " from being counted as different products
df['StockCode'] = df['StockCode'].astype(str).str.strip()

# 2. Filter out non-product items (like POSTAGE, BANK CHARGES, DOTCOM POSTAGE)
# These aren't real products we want to recommend
bad_codes = ['POST', 'DOT', 'C2', 'M', 'BANK CHARGES', 'PADS', 'CRUK']
df = df[~df['StockCode'].isin(bad_codes)]

Preparing data...


In [7]:
# Filter Users
# Calculating similarity for 4,000+ users can be slow and memory-heavy.
# Filter to the TOP 1,000 users by Total Spend to keep the demo fast.
print("Filtering to top 1000 active users for performance...")

top_users = df.groupby('CustomerID')['Total_Spend'].sum().sort_values(ascending=False).head(1000).index
df_filtered = df[df['CustomerID'].isin(top_users)]

print(f"Data reduced to {df_filtered.shape[0]} rows for {len(top_users)} users.")

Filtering to top 1000 active users for performance...
Data reduced to 245056 rows for 1000 users.


In [8]:
# Create the user-item matrix
print("Building User-Item Matrix...")

# Rows = CustomerID, Columns = StockCode, Values = Quantity
user_item_matrix = df_filtered.pivot_table(
    index='CustomerID', 
    columns='StockCode', 
    values='Quantity', 
    aggfunc='sum', 
    fill_value=0
)

Building User-Item Matrix...


In [10]:
# Calculate similiarity
print("Calculating User Similarity ...")

# This creates a matrix where Row A, Col B = "How similar is User A to User B?"
user_similarity = cosine_similarity(user_item_matrix)

# Convert to DataFrame so we can use UserIDs as index/columns later
user_sim_df = pd.DataFrame(
    user_similarity, 
    index=user_item_matrix.index, 
    columns=user_item_matrix.index
)

Calculating User Similarity ...


In [11]:
# Create va product lookup dictionary
# Need this to convert '85123A' back to 'White T-Light Holder'
product_descriptions = df_filtered.drop_duplicates('StockCode').set_index('StockCode')['Description'].to_dict()

In [13]:
print("Saving models...")

os.makedirs('../models', exist_ok=True)

# 1. The Similarity Matrix (The "Brain")
joblib.dump(user_sim_df, '../models/recommendation_engine.pkl')

# 2. The User-Item Matrix (The "Map" - who bought what)
joblib.dump(user_item_matrix, '../models/user_item_matrix.pkl')

# 3. The Product Names (The "Translator")
joblib.dump(product_descriptions, '../models/product_lookup.pkl')

print("Recommendation Engine built successfully!")
print("Files saved:")
print(" - recommendation_engine.pkl")
print(" - user_item_matrix.pkl")
print(" - product_lookup.pkl")

Saving models...
Recommendation Engine built successfully!
Files saved:
 - recommendation_engine.pkl
 - user_item_matrix.pkl
 - product_lookup.pkl
